In [ ]:
import math
import itertools
import pandas as pd

products = [1, 2, 3, 4]
D = [1000, 300, 100, 50]
c = 150
s = [20, 25, 30, 50]
u = [50, 60, 30, 30]
r = 0.15

h = [r * ui for ui in u]

df = pd.DataFrame({
    "Product": products,
    "Demand_D": D,
    "Specific_order_cost_s": s,
    "Unit_cost_u": u,
    "Holding_cost_h (=r*u)": h
})

df

,Product,Demand_D,Specific_order_cost_s,Unit_cost_u,Holding_cost_h (=r*u)
0,1,1000,20,50,7.5
1,2,300,25,60,9.0
2,3,100,30,30,4.5
3,4,50,50,30,4.5


In [ ]:
def eoq_cost(Di, Ki, hi):
    """
    Min annual cost for classic EOQ:
    ordering = Ki * Di / Q
    holding  = hi * Q / 2
    at optimum, total = sqrt(2*Di*Ki*hi)
    """
    return math.sqrt(2 * Di * Ki * hi)

def strategy_1_independent(D, h, s, c):
    """
    Each product ordered independently.
    Each order pays (common + specific) because orders are separate.
    """
    total_cost = 0.0
    details = []
    for i, (Di, hi, si) in enumerate(zip(D, h, s), start=1):
        Ki = c + si
        cost_i = eoq_cost(Di, Ki, hi)
        total_cost += cost_i

        # EOQ quantity
        Q_star = math.sqrt(2 * Di * Ki / hi)
        details.append([i, Ki, Q_star, cost_i])

    det_df = pd.DataFrame(details, columns=["Product", "K_i (c+s)", "EOQ_Q*", "Annual_cost"])
    return total_cost, det_df

def strategy_2_same_frequency(D, h, s, c):
    """
    All products ordered every T years (same frequency).
    Cost(T) = (c + sum s_i)/T + sum (h_i * D_i * T)/2
    Optimize => T* = sqrt(A/B), cost* = 2*sqrt(A*B)
    where A = c + sum s_i, B = sum(h_i*D_i)/2
    """
    A = c + sum(s)
    B = sum(hi * Di for hi, Di in zip(h, D)) / 2
    T_star = math.sqrt(A / B)
    cost_star = 2 * math.sqrt(A * B)

    # quantities ordered each time
    Qs = [Di * T_star for Di in D]

    det_df = pd.DataFrame({
        "Product": products,
        "Order_qty_Qi (=Di*T*)": Qs
    })
    return cost_star, T_star, det_df

def strategy_3_tailored_aggregation(D, h, s, c, powers=(1,2,4,8,16,32,64)):
    """
    Tailored aggregation:
    - base cycle time T0
    - product i ordered every k_i * T0, with k_i in powers of 2 (searched by brute force)

    Cost(T0, k) = c/T0 + sum [ s_i/(k_i*T0) + (h_i*D_i*(k_i*T0))/2 ]

    For fixed k, optimize over T0:
      A = c + sum(s_i/k_i)
      B = sum(h_i*D_i*k_i)/2
      T0* = sqrt(A/B)
      cost* = 2*sqrt(A*B)
    """
    best = None

    for ks in itertools.product(powers, repeat=len(D)):
        A = c + sum(si / ki for si, ki in zip(s, ks))
        B = sum(hi * Di * ki for hi, Di, ki in zip(h, D, ks)) / 2
        cost = 2 * math.sqrt(A * B)
        if best is None or cost < best["cost"]:
            T0_star = math.sqrt(A / B)
            best = {
                "cost": cost,
                "ks": ks,
                "T0_star": T0_star,
                "A": A,
                "B": B
            }


    ks = best["ks"]
    T0 = best["T0_star"]
    order_interval = [ki * T0 for ki in ks]
    order_qty = [Di * ki * T0 for Di, ki in zip(D, ks)]

    det_df = pd.DataFrame({
        "Product": products,
        "k_i (multiple of base cycle)": ks,
        "Order_interval_years (=k_i*T0*)": order_interval,
        "Order_qty (=D_i*k_i*T0*)": order_qty
    })
    return best["cost"], best["T0_star"], best["ks"], det_df


# Run all 3 strategies
cost1, det1 = strategy_1_independent(D, h, s, c)
cost2, T2, det2 = strategy_2_same_frequency(D, h, s, c)
cost3, T0, ks, det3 = strategy_3_tailored_aggregation(D, h, s, c)

cost1, cost2, cost3

(3271.475282978272, 2445.65942027912, 2310.8440016582686)

In [ ]:
print("=== Strategy 1: Independent sourcing ===")
display(det1)
print(f"Total annual operational cost: {cost1:.2f}")

print("=== Strategy 2: Same frequency for all ===")
print(f"Optimal cycle time T*: {T2:.6f} years  (~{T2*365:.1f} days)")
display(det2)
print(f"Total annual operational cost: {cost2:.2f}")

print("=== Strategy 3: Tailored aggregation ===")
print(f"Best k's (k1,k2,k3,k4): {ks}")
print(f"Optimal base cycle time T0*: {T0:.6f} years  (~{T0*365:.1f} days)")
display(det3)
print(f"Total annual operational cost: {cost3:.2f}")

=== Strategy 1: Independent sourcing ===


,Product,K_i (c+s),EOQ_Q*,Annual_cost
0,1,170,212.916259,1596.871942
1,2,175,108.012345,972.111105
2,3,180,89.442719,402.492236
3,4,200,66.666667,300.000000


Total annual operational cost: 3271.48

=== Strategy 2: Same frequency for all ===
Optimal cycle time T*: 0.224888 years  (~82.1 days)


,Product,Order_qty_Qi (=Di*T*)
0,1,224.888223
1,2,67.466467
2,3,22.488822
3,4,11.244411


Total annual operational cost: 2445.66

=== Strategy 3: Tailored aggregation ===
Best k's (k1,k2,k3,k4): (1, 1, 2, 4)
Optimal base cycle time T0*: 0.192570 years  (~70.3 days)


,Product,k_i (multiple of base cycle),Order_interval_years (=k_i*T0*),Order_qty (=D_i*k_i*T0*)
0,1,1,0.192570,192.570333
1,2,1,0.192570,57.771100
2,3,2,0.385141,38.514067
3,4,4,0.770281,38.514067


Total annual operational cost: 2310.84


###When each product is ordered independently, the company ends up paying the common ordering cost every single time an order is placed, which makes this option the most expensive at 3271.48 per year. If all four products are ordered together at the same frequency, the common ordering cost is shared across items, which lowers the total annual cost to 2445.66. However, the most efficient approach is the tailored aggregation strategy. In this case, higher-demand products (1 and 2) are ordered more frequently, while lower-demand products (3 and 4) are ordered less often by using multiples of a base cycle time. This better balances ordering and holding costs across products. As a result, the tailored aggregation strategy achieves the lowest total annual operational cost of 2310.84, making it the optimal sourcing strategy.